# Starting qdrant instance

```shell
docker run -d -p 6333:6333 -p 6334:6334 \
  -v $(pwd)/qdrant_storage:/qdrant/storage \
  --name qdrant-server \
  qdrant/qdrant
```

In [10]:
from mem0 import Memory

config = {
    "vector_store": {
        "provider": "qdrant",
        "config": {"collection_name": "test", "host": "localhost", "port": 6333, "embedding_model_dims": 768},
    },
    "llm": {
        "provider": "ollama",
        "config": {"model": "llama3:8b", "temperature": 0, "max_tokens": 2000, "ollama_base_url": "http://localhost:11434"},
    },
    "embedder": {
        "provider": "ollama",
        "config": {"model": "nomic-embed-text:latest", "ollama_base_url": "http://localhost:11434"},
    },
}
m = Memory.from_config(config)
m.add("I'm visiting Paris", user_id="john")

{'results': []}

In [11]:
m.search("Where am I visiting?", filters={"user_id": "john"})

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

{'results': [{'id': 'c96229b7-c2d0-42cc-9cea-849523dfd98e',
   'memory': 'User is planning to visit Paris',
   'hash': '2a4d85a4a98cbbb2a9e8d03dd54ad749',
   'metadata': None,
   'score': 0.6299447,
   'created_at': '2026-08-30T15:26:35.711940+00:00',
   'updated_at': '2026-08-30T15:26:35.711940+00:00',
   'user_id': 'john',
   'attributed_to': 'user'},
  {'id': '3d09d550-31c7-4430-96db-2881b1ea0a50',
   'memory': 'User is visiting Paris around August 30, 2026',
   'hash': '8bc4407474d6ec49533a3d27ccb6a055',
   'metadata': None,
   'score': 0.6087637,
   'created_at': '2026-08-30T15:29:00.052263+00:00',
   'updated_at': '2026-08-30T15:29:00.052263+00:00',
   'user_id': 'john'}]}

## Evaluating Time

In [12]:
%%time
m.add("I will be starting tomorrow", user_id="john", )

CPU times: user 147 ms, sys: 106 ms, total: 254 ms
Wall time: 13.5 s


{'results': [{'id': 'b6763cdf-974b-4cf9-aba0-8f47f132ce56',
   'memory': 'User is starting their Paris trip tomorrow',
   'event': 'ADD'}]}

In [14]:
%%time

m.search("when will I be starting?", filters={"user_id": "john"})

CPU times: user 7.47 ms, sys: 4.08 ms, total: 11.5 ms
Wall time: 50.3 ms


{'results': [{'id': 'b6763cdf-974b-4cf9-aba0-8f47f132ce56',
   'memory': 'User is starting their Paris trip tomorrow',
   'hash': 'd0b718149687b4cb45392ef96618c197',
   'metadata': None,
   'score': 0.2782753901346216,
   'created_at': '2026-08-30T16:00:24.688847+00:00',
   'updated_at': '2026-08-30T16:00:24.688847+00:00',
   'user_id': 'john',
   'attributed_to': 'user'},
  {'id': '3d09d550-31c7-4430-96db-2881b1ea0a50',
   'memory': 'User is visiting Paris around August 30, 2026',
   'hash': '8bc4407474d6ec49533a3d27ccb6a055',
   'metadata': None,
   'score': 0.25428018,
   'created_at': '2026-08-30T15:29:00.052263+00:00',
   'updated_at': '2026-08-30T15:29:00.052263+00:00',
   'user_id': 'john'},
  {'id': 'c96229b7-c2d0-42cc-9cea-849523dfd98e',
   'memory': 'User is planning to visit Paris',
   'hash': '2a4d85a4a98cbbb2a9e8d03dd54ad749',
   'metadata': None,
   'score': 0.2136676,
   'created_at': '2026-08-30T15:26:35.711940+00:00',
   'updated_at': '2026-08-30T15:26:35.711940+00:00'

## Filter Pattens

[Filter Patterns](https://docs.mem0.ai/core-concepts/memory-operations/search#filter-patterns), we should have it customize for the genie

## when to add Memory

Add memory whenever your agent learns something useful:
* A new user preference is shared
* A decision or suggestion is made
* A goal or task is completed
* A new entity is introduced
* A user gives feedback or clarification

## Memory Router (Example 1)

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class RobustMemoryRouter:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", threshold: float = 0.35):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold
        
        self.intent_examples = {
            "ADD": [
                "Remember that our API gateway port is 8443",
                "Remember that my favorite language is Python",
                "Store this information: the project deadline is Friday",
                "Save this note about the database credentials",
                "Keep in mind that John is the lead engineer",
                "Take note that the server IP is 192.168.1.50",
                "Add this to my memory: my wife's birthday is June 4th",
                "Please save this fact for later",
                "Remember this detail"
            ],
            "SEARCH": [
                "What is the server IP address?",
                "What port does the API gateway run on?",
                "Do you remember when the deadline is?",
                "What did I say about John's role?",
                "Look up my favorite programming language",
                "Find the note about database credentials",
                "When is my wife's birthday",
                "What is saved in memory about the database?",
                "Retrieve my notes on the project"
            ]
        }
        
        # Store individual normalized embeddings
        self.intent_embeddings = {
            intent: self.model.encode(examples, normalize_embeddings=True)
            for intent, examples in self.intent_examples.items()
        }

    def route(self, query: str) -> dict:
        query_embedding = self.model.encode([query], normalize_embeddings=True)[0]
        
        # 1. Fast keyword override for unambiguous commands
        q_lower = query.lower().strip()
        add_prefixes = ("remember that", "remember:", "save this", "store this", "take note", "add to memory")
        if any(q_lower.startswith(p) for p in add_prefixes):
            return {
                "action": "ADD",
                "confidence": 1.0,
                "payload": self._extract_payload(query, "ADD"),
                "method": "rule_override"
            }

        # 2. Max Cosine Similarity across all exemplars in each class
        scores = {}
        for intent, emb_matrix in self.intent_embeddings.items():
            sims = cosine_similarity([query_embedding], emb_matrix)[0]
            scores[intent] = float(np.max(sims))  # Best matching exemplar score

        best_intent = max(scores, key=scores.get)
        confidence = scores[best_intent]

        # Default to DIRECT if confidence is below threshold
        if confidence < self.threshold:
            action = "DIRECT"
        else:
            action = best_intent

        return {
            "action": action,
            "confidence": round(confidence, 3),
            "payload": self._extract_payload(query, action),
            "scores": {k: round(v, 3) for k, v in scores.items()}
        }

    def _extract_payload(self, query: str, intent: str) -> str:
        if intent == "ADD":
            prefixes = [
                "remember that ", "remember ", "save this: ", "save this ",
                "store this: ", "store this ", "take note that ", "add to memory: "
            ]
            q_lower = query.lower()
            for prefix in prefixes:
                if q_lower.startswith(prefix):
                    return query[len(prefix):].strip()
        return query

In [22]:
router = RobustMemoryRouter()

queries = [
    "Remember that our API gateway port is 8443",
    "What port does the API gateway run on?",
    "Write a quicksort implementation in Rust"
]

for q in queries:
    decision = router.route(q)
    print(f"Query: \"{q}\" -> Action: {decision['action']} (Confidence: {decision['confidence']})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: "Remember that our API gateway port is 8443" -> Action: ADD (Confidence: 1.0)
Query: "What port does the API gateway run on?" -> Action: SEARCH (Confidence: 1.0)
Query: "Write a quicksort implementation in Rust" -> Action: DIRECT (Confidence: 0.2)
